In [91]:
import os 
cwd = os.getcwd()
cwd

os.chdir("/Users/jmthomas565/Desktop/Education/Applied Stats MSc/MSc Project/Git Files/msc-project-source-code-files-24-25-jmthomas565/Code/Project Code Pipeline")

new_cwd = os.getcwd() 

new_cwd


'/Users/jmthomas565/Desktop/Education/Applied Stats MSc/MSc Project/Git Files/msc-project-source-code-files-24-25-jmthomas565/Code/Project Code Pipeline'

In [93]:
%run 00_inititial_transformations.ipynb

['2017-07-01', '2017-07-02', '2017-07-03', '2017-07-04', '2017-07-05'] ... ['2022-09-27', '2022-09-28', '2022-09-29', '2022-09-30', '2022-10-01']


### Making Variables Stationary

In [95]:
X_train.head()

,branded_search_spend,nonbranded_search_spend,facebook_spend,print_spend,ooh_spend,tv_spend,radio_spend,gdp,base_rate,UNRATENSA_daily
Week,,,,,,,,,,
2017-07-23,1528.8,463.32,802.620,0,0,0,0,4958224,1.15,4.6
2017-07-30,1575.6,468.00,819.312,0,0,0,0,4958224,1.15,4.6
2017-08-06,1544.4,477.36,749.034,0,0,0,0,4958224,1.16,4.5
2017-08-13,1528.8,468.00,741.468,0,0,0,0,4958224,1.16,4.5
2017-08-20,1560.0,458.64,811.200,0,0,0,0,4958224,1.16,4.5


In [97]:
X_train_stationary = X_train

y_train_stationary = y_train

In [99]:
#Taking 1st difference of the variables

X_train_stationary['nonbranded_search_spend'] = X_train['nonbranded_search_spend'].diff()

X_train_stationary['ooh_spend'] = X_train['ooh_spend'].diff()

X_train_stationary['tv_spend'] = X_train['tv_spend'].diff()

y_train_stationary['sales'] = y_train['sales'].diff()


In [101]:
#Removing the 1st row due to nulls from the differencing

X_train_stationary = X_train_stationary.iloc[1:]

y_train_stationary = y_train_stationary.iloc[1:]

X_train_stationary.head()

,branded_search_spend,nonbranded_search_spend,facebook_spend,print_spend,ooh_spend,tv_spend,radio_spend,gdp,base_rate,UNRATENSA_daily
Week,,,,,,,,,,
2017-07-30,1575.6,4.68,819.312,0,0.0,0.0,0,4958224,1.15,4.6
2017-08-06,1544.4,9.36,749.034,0,0.0,0.0,0,4958224,1.16,4.5
2017-08-13,1528.8,-9.36,741.468,0,0.0,0.0,0,4958224,1.16,4.5
2017-08-20,1560.0,-9.36,811.200,0,0.0,0.0,0,4958224,1.16,4.5
2017-08-27,1606.8,14.04,827.502,0,0.0,0.0,0,4958224,1.16,4.5


In [73]:
y_train_stationary.tail()

,sales
Week,
2021-06-13,2009.70
2021-06-20,-2744.28
2021-06-27,-651.42
2021-07-04,1386.00
2021-07-11,-734.58


### Running Regression with Stationary Variables

In [75]:
import pandas as pd
import numpy as np
from sklearn.linear_model import Ridge
from sklearn.metrics import r2_score


In [89]:
y_train.isna().sum()

sales    1
dtype: int64

In [103]:

#Cross-validation hold-out
hold_out_split = -26
y_train_calib = y_train_stationary.iloc[:hold_out_split]
y_train_hold_out = y_train_stationary.iloc[hold_out_split:]


#Defining adj_R^2 function
def adj_r2(y_true, y_pred, p):
    r2 = r2_score(y_true, y_pred)
    n = len(y_true)
    #Making sure the denominator of adj R^2 > 0
    if n - p - 1 <= 0:
        return np.nan
    calculation = 1 - (1 - r2) * (n - 1) / (n - p - 1)
    return calculation

In [109]:

def test_model_adj_r2(X_train, regressors):
    ridge = Ridge(alpha=1.0, positive = True)
    
    #splitting for hold-out cv
    X_train_calib = X_train.iloc[:hold_out_split]
    X_val_hold_out = X_train.iloc[hold_out_split:]

    ridge.fit(X_train_calib, y_train_calib)
    y_pred = ridge.predict(X_val_hold_out)
    
    score = adj_r2(y_train_hold_out, y_pred, p=X_train_calib.shape[1])
    print(f"{regressors}: Adjusted R2 = {round(score, 4)}")

#testing different numbers of regressors, adding regressors iteratively, one by one
X_1reg = X_train_stationary.iloc[:, :1]
test_model_adj_r2(X_1reg, "1 regressors")

X_2reg = X_train_stationary.iloc[:, :2]
test_model_adj_r2(X_2reg, "2 regressors")

X_3reg = X_train_stationary.iloc[:, :3]  
test_model_adj_r2(X_3reg, "3 regressors")

X_4reg = X_train_stationary.iloc[:, :4]
test_model_adj_r2(X_4reg, "4 regressors")

X_5reg = X_train_stationary.iloc[:, :5]
test_model_adj_r2(X_5reg, "5 regressors")

X_6reg = X_train_stationary.iloc[:, :6]
test_model_adj_r2(X_6reg, "6 regressors")

X_7reg = X_train_stationary.iloc[:, :7]
test_model_adj_r2(X_7reg, "7 regressors")


1 regressors: Adjusted R2 = -0.0498
2 regressors: Adjusted R2 = -0.0955
3 regressors: Adjusted R2 = -0.1511
4 regressors: Adjusted R2 = -0.2004
5 regressors: Adjusted R2 = -0.2604
6 regressors: Adjusted R2 = -0.3168
7 regressors: Adjusted R2 = -0.39
